# CHIRPS Rainfall Data Cleaning & Validation

Notebook ini digunakan untuk melakukan validasi dan pemeriksaan kualitas data curah hujan CHIRPS v3.0 SAT periode 1–30 September 2026.

Proses meliputi pemeriksaan kelengkapan file, metadata raster, kualitas nilai piksel, dan persiapan data untuk cropping wilayah Kalimantan.

In [2]:

from pathlib import Path
from datetime import date, timedelta

import rasterio
import numpy as np
import pandas as pd


In [3]:
data_path = Path("../../data/raw/chirps")
boundary_path = Path("../../data/processed/boundaries/kalimantan_provinces.geojson")

tif_files = sorted(data_path.glob("*.tif"))

print("Jumlah file TIFF:", len(tif_files))
print("Boundary tersedia:", boundary_path.exists())

Jumlah file TIFF: 30
Boundary tersedia: True


## 1. File Completeness Validation

Memeriksa kelengkapan 30 file raster CHIRPS berdasarkan periode penelitian, yaitu 1–30 September 2026.

In [4]:
start_date = date(2026, 9, 1)
end_date = date(2026, 9, 30)

expected_dates = {
    (start_date + timedelta(days=i)).strftime("%Y.%m.%d")
    for i in range((end_date - start_date).days + 1)
}

actual_dates = {
    file.name.replace("chirps-v3.0.sat.", "").replace(".tif", "")
    for file in tif_files
}

missing_dates = sorted(expected_dates - actual_dates)
unexpected_dates = sorted(actual_dates - expected_dates)

print("Expected files:", len(expected_dates))
print("Available files:", len(actual_dates))
print("Missing dates:", missing_dates)
print("Unexpected dates:", unexpected_dates)

if len(tif_files) == 30 and not missing_dates and not unexpected_dates:
    print("STATUS: VALID")
else:
    print("STATUS: INCOMPLETE")

Expected files: 30
Available files: 30
Missing dates: []
Unexpected dates: []
STATUS: VALID


In [5]:
file_info = []

for file in tif_files:
    file_info.append({
        "filename": file.name,
        "size_mb": round(file.stat().st_size / (1024 ** 2), 2)
    })

file_size_df = pd.DataFrame(file_info)

display(file_size_df)

print("Total size:",
      round(file_size_df["size_mb"].sum(), 2), "MB")

,filename,size_mb
0,chirps-v3.0.sat.2026.09.01.tif,14.53
1,chirps-v3.0.sat.2026.09.02.tif,14.88
2,chirps-v3.0.sat.2026.09.03.tif,15.18
3,chirps-v3.0.sat.2026.09.04.tif,15.36
4,chirps-v3.0.sat.2026.09.05.tif,15.61
5,chirps-v3.0.sat.2026.09.06.tif,15.05
6,chirps-v3.0.sat.2026.09.07.tif,14.89
7,chirps-v3.0.sat.2026.09.08.tif,14.54
8,chirps-v3.0.sat.2026.09.09.tif,14.85
9,chirps-v3.0.sat.2026.09.10.tif,14.85


Total size: 448.72 MB


## 2. Raster Metadata Inspection

Memeriksa karakteristik spasial dan teknis raster CHIRPS, meliputi CRS, resolusi, dimensi, jumlah band, tipe data, dan nilai NoData.

In [6]:
sample_file = tif_files[0]

with rasterio.open(sample_file) as src:
    print("Filename:", sample_file.name)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("Data type:", src.dtypes)
    print("NoData:", src.nodata)
    print("Bounds:", src.bounds)

Filename: chirps-v3.0.sat.2026.09.01.tif
CRS: EPSG:4326
Resolution: (0.05000000074505806, 0.05000000074505806)
Width: 7200
Height: 2400
Bands: 1
Data type: ('float32',)
NoData: None
Bounds: BoundingBox(left=-180.0, bottom=-60.00000178813934, right=180.00000536441803, top=60.0)


## 3. Raster Data Quality Check

Memeriksa nilai piksel curah hujan pada raster sampel untuk mengetahui rentang nilai, rata-rata, jumlah piksel valid, NoData, dan nilai negatif.

In [10]:

with rasterio.open(sample_file) as src:
    rainfall = src.read(1)

    # Mask nilai khusus yang menandakan data hilang
    rainfall = np.ma.masked_equal(rainfall, -9999.0)

    valid_values = rainfall.compressed()

    print("===== RASTER DATA QUALITY =====")
    print("Minimum rainfall:", valid_values.min())
    print("Maximum rainfall:", valid_values.max())
    print("Mean rainfall:", valid_values.mean())
    print("Valid pixels:", len(valid_values))
    print("Missing pixels:", rainfall.mask.sum())
    print("Negative rainfall:", (valid_values < 0).sum())


===== RASTER DATA QUALITY =====
Minimum rainfall: 0.0
Maximum rainfall: 231.66742
Mean rainfall: 2.143
Valid pixels: 4848282
Missing pixels: 12431718
Negative rainfall: 0


In [11]:

missing_summary = []

for file in tif_files:
    with rasterio.open(file) as src:
        rainfall = src.read(1)

        missing_count = np.sum(rainfall == -9999.0)
        valid_count = np.sum(rainfall != -9999.0)

        missing_summary.append({
            "date": file.stem.split(".")[-3:],
            "missing_pixels": int(missing_count),
            "valid_pixels": int(valid_count),
            "missing_percentage": round(
                missing_count / rainfall.size * 100, 2
            )
        })

missing_df = pd.DataFrame(missing_summary)

display(missing_df)
print("Total missing pixels:",
      missing_df["missing_pixels"].sum())


,date,missing_pixels,valid_pixels,missing_percentage
0,"[2026, 09, 01]",12431718,4848282,71.94
1,"[2026, 09, 02]",12431718,4848282,71.94
2,"[2026, 09, 03]",12431718,4848282,71.94
3,"[2026, 09, 04]",12431718,4848282,71.94
4,"[2026, 09, 05]",12431718,4848282,71.94
5,"[2026, 09, 06]",12431718,4848282,71.94
6,"[2026, 09, 07]",12431718,4848282,71.94
7,"[2026, 09, 08]",12431718,4848282,71.94
8,"[2026, 09, 09]",12431718,4848282,71.94
9,"[2026, 09, 10]",12431718,4848282,71.94


Total missing pixels: 372951715


In [12]:
metadata_list = []

for file in tif_files:
    with rasterio.open(file) as src:
        metadata_list.append({
            "filename": file.name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution": src.res,
            "bands": src.count,
            "dtype": src.dtypes[0],
            "nodata": src.nodata
        })

metadata_df = pd.DataFrame(metadata_list)

display(metadata_df)

print("Unique CRS:", metadata_df["crs"].nunique())
print("Unique dimensions:",
      metadata_df["width"].nunique(),
      metadata_df["height"].nunique())
print("Unique resolutions:", metadata_df["resolution"].nunique())

,filename,crs,width,height,resolution,bands,dtype,nodata
0,chirps-v3.0.sat.2026.09.01.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
1,chirps-v3.0.sat.2026.09.02.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
2,chirps-v3.0.sat.2026.09.03.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
3,chirps-v3.0.sat.2026.09.04.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
4,chirps-v3.0.sat.2026.09.05.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
5,chirps-v3.0.sat.2026.09.06.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
6,chirps-v3.0.sat.2026.09.07.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
7,chirps-v3.0.sat.2026.09.08.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
8,chirps-v3.0.sat.2026.09.09.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None
9,chirps-v3.0.sat.2026.09.10.tif,EPSG:4326,7200,2400,"(0.05000000074505806, 0.05000000074505806)",1,float32,None


Unique CRS: 1
Unique dimensions: 1 1
Unique resolutions: 1


## 4. Validation Summary

Merangkum hasil pemeriksaan dataset CHIRPS untuk memastikan kelengkapan file dan konsistensi metadata sebelum proses cropping dan transformation.

In [9]:
print("===== CHIRPS VALIDATION SUMMARY =====")

print("Total TIFF files:", len(tif_files))
print("Missing dates:", len(missing_dates))
print("Unexpected dates:", len(unexpected_dates))

print("CRS consistent:", metadata_df["crs"].nunique() == 1)
print("Resolution consistent:",
      metadata_df["resolution"].nunique() == 1)

print("Validation completed.")

===== CHIRPS VALIDATION SUMMARY =====
Total TIFF files: 30
Missing dates: 0
Unexpected dates: 0
CRS consistent: True
Resolution consistent: True
Validation completed.


## 5. Spatial Cropping

Melakukan pemotongan raster CHIRPS berdasarkan batas administratif lima provinsi di Pulau Kalimantan.

Proses ini bertujuan membatasi cakupan data pada wilayah penelitian dan mengurangi ukuran file. Nilai NoData tetap dipertahankan dan ditetapkan secara eksplisit menggunakan nilai -9999.

In [13]:
import geopandas as gpd
from rasterio.mask import mask

# Baca batas administratif Kalimantan
gdf_boundary = gpd.read_file(boundary_path)

print("Jumlah provinsi:", len(gdf_boundary))
print("CRS boundary:", gdf_boundary.crs)
print("CRS raster:", rasterio.open(tif_files[0]).crs)

print("\nNama provinsi:")
print(gdf_boundary["namobj"].tolist())

Jumlah provinsi: 5
CRS boundary: EPSG:4326
CRS raster: EPSG:4326

Nama provinsi:
['Ketapang', 'Kotabaru', 'Katingan', 'Paser', 'Bulungan']


In [14]:
# Folder output
output_path = Path("../../data/processed/chirps/daily")
output_path.mkdir(parents=True, exist_ok=True)

# Samakan CRS boundary dengan CRS raster
with rasterio.open(tif_files[0]) as src:
    boundary = gdf_boundary.to_crs(src.crs)

# Ambil geometri provinsi
geometries = [
    geom.__geo_interface__
    for geom in boundary.geometry
    if geom is not None and not geom.is_empty
]

crop_summary = []

for file in tif_files:

    with rasterio.open(file) as src:

        # Crop raster berdasarkan boundary
        cropped, cropped_transform = mask(
            src,
            geometries,
            crop=True,
            nodata=-9999.0,
            filled=True
        )

        # Perbarui metadata
        metadata = src.meta.copy()
        metadata.update({
            "height": cropped.shape[1],
            "width": cropped.shape[2],
            "transform": cropped_transform,
            "nodata": -9999.0,
            "compress": "deflate"
        })

    # Simpan hasil cropping
    output_file = output_path / file.name

    with rasterio.open(output_file, "w", **metadata) as dst:
        dst.write(cropped)

    # Hitung piksel valid
    valid_pixels = cropped[0][cropped[0] != -9999.0]

    crop_summary.append({
        "filename": file.name,
        "width": cropped.shape[2],
        "height": cropped.shape[1],
        "valid_pixels": len(valid_pixels),
        "missing_pixels": int(
            np.sum(cropped[0] == -9999.0)
        )
    })

    print("Processed:", file.name)

crop_df = pd.DataFrame(crop_summary)

print("\nCropping selesai!")
display(crop_df)

Processed: chirps-v3.0.sat.2026.09.01.tif
Processed: chirps-v3.0.sat.2026.09.02.tif
Processed: chirps-v3.0.sat.2026.09.03.tif
Processed: chirps-v3.0.sat.2026.09.04.tif
Processed: chirps-v3.0.sat.2026.09.05.tif
Processed: chirps-v3.0.sat.2026.09.06.tif
Processed: chirps-v3.0.sat.2026.09.07.tif
Processed: chirps-v3.0.sat.2026.09.08.tif
Processed: chirps-v3.0.sat.2026.09.09.tif
Processed: chirps-v3.0.sat.2026.09.10.tif
Processed: chirps-v3.0.sat.2026.09.11.tif
Processed: chirps-v3.0.sat.2026.09.12.tif
Processed: chirps-v3.0.sat.2026.09.13.tif
Processed: chirps-v3.0.sat.2026.09.14.tif
Processed: chirps-v3.0.sat.2026.09.15.tif
Processed: chirps-v3.0.sat.2026.09.16.tif
Processed: chirps-v3.0.sat.2026.09.17.tif
Processed: chirps-v3.0.sat.2026.09.18.tif
Processed: chirps-v3.0.sat.2026.09.19.tif
Processed: chirps-v3.0.sat.2026.09.20.tif
Processed: chirps-v3.0.sat.2026.09.21.tif
Processed: chirps-v3.0.sat.2026.09.22.tif
Processed: chirps-v3.0.sat.2026.09.23.tif
Processed: chirps-v3.0.sat.2026.09

,filename,width,height,valid_pixels,missing_pixels
0,chirps-v3.0.sat.2026.09.01.tif,210,188,17373,22107
1,chirps-v3.0.sat.2026.09.02.tif,210,188,17373,22107
2,chirps-v3.0.sat.2026.09.03.tif,210,188,17373,22107
3,chirps-v3.0.sat.2026.09.04.tif,210,188,17373,22107
4,chirps-v3.0.sat.2026.09.05.tif,210,188,17373,22107
5,chirps-v3.0.sat.2026.09.06.tif,210,188,17373,22107
6,chirps-v3.0.sat.2026.09.07.tif,210,188,17373,22107
7,chirps-v3.0.sat.2026.09.08.tif,210,188,17373,22107
8,chirps-v3.0.sat.2026.09.09.tif,210,188,17373,22107
9,chirps-v3.0.sat.2026.09.10.tif,210,188,17373,22107


In [15]:
processed_files = sorted(output_path.glob("*.tif"))

print("===== CROPPING VALIDATION =====")
print("Jumlah raster hasil:", len(processed_files))
print("Folder output:", output_path.resolve())

with rasterio.open(processed_files[0]) as src:
    rainfall = src.read(1, masked=True)

    valid_values = rainfall.compressed()

    print("\nSample raster:", processed_files[0].name)
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Dimensions:", src.width, "x", src.height)
    print("NoData:", src.nodata)
    print("Bounds:", src.bounds)

    print("\nMinimum rainfall:", valid_values.min())
    print("Maximum rainfall:", valid_values.max())
    print("Mean rainfall:", valid_values.mean())
    print("Valid pixels:", len(valid_values))
    print("Missing pixels:", rainfall.mask.sum())

===== CROPPING VALIDATION =====
Jumlah raster hasil: 30
Folder output: D:\Project\karhutla-spatial-temporal-analysis\data\processed\chirps\daily

Sample raster: chirps-v3.0.sat.2026.09.01.tif
CRS: EPSG:4326
Resolution: (0.05000000074505806, 0.05000000074505806)
Dimensions: 210 x 188
NoData: -9999.0
Bounds: BoundingBox(left=108.55000429973006, bottom=-4.9500009678304195, right=119.05000445619226, top=4.449999172240496)

Minimum rainfall: 0.0
Maximum rainfall: 34.78628
Mean rainfall: 0.502578
Valid pixels: 17373
Missing pixels: 22107


In [16]:
from rasterio.mask import mask
import numpy as np
import rasterio

sample_file = processed_files[0]

with rasterio.open(sample_file) as src:
    boundary = gdf_boundary.to_crs(src.crs)

    geometries = [
        geom.__geo_interface__
        for geom in boundary.geometry
        if geom is not None and not geom.is_empty
    ]

    with rasterio.open(tif_files[0]) as original:
        cropped_masked, transform = mask(
            original,
            geometries,
            crop=True,
            filled=False
        )

arr = cropped_masked[0]

# Area di luar polygon provinsi
outside_mask = np.ma.getmaskarray(arr)

# NoData asli di dalam polygon
original_missing = (
    (arr.data == -9999.0) & ~outside_mask
)

# Piksel valid di dalam polygon
inside_valid = (
    ~outside_mask & ~original_missing
)

valid_values = arr.data[inside_valid]

print("===== DATA QUALITY INSIDE KALIMANTAN =====")
print("Pixels inside province:", int((~outside_mask).sum()))
print("Valid pixels:", int(inside_valid.sum()))
print("Missing pixels inside province:", int(original_missing.sum()))

print("Minimum rainfall:", valid_values.min())
print("Maximum rainfall:", valid_values.max())
print("Mean rainfall:", valid_values.mean())

===== DATA QUALITY INSIDE KALIMANTAN =====
Pixels inside province: 17373
Valid pixels: 17373
Missing pixels inside province: 0
Minimum rainfall: 0.0
Maximum rainfall: 34.78628
Mean rainfall: 0.502578


In [17]:
quality_summary = []

for file in tif_files:
    with rasterio.open(file) as src:
        cropped_masked, _ = mask(
            src,
            geometries,
            crop=True,
            filled=False
        )

    arr = cropped_masked[0]

    outside_mask = np.ma.getmaskarray(arr)

    original_missing = (
        (arr.data == -9999.0) & ~outside_mask
    )

    inside_valid = (
        ~outside_mask & ~original_missing
    )

    quality_summary.append({
        "date": file.stem.split(".")[-1],
        "valid_pixels": int(inside_valid.sum()),
        "missing_pixels": int(original_missing.sum()),
        "missing_percentage": (
            original_missing.sum()
            / (~outside_mask).sum()
        ) * 100
    })

quality_df = pd.DataFrame(quality_summary)

display(quality_df)

print("===== OVERALL QUALITY =====")
print("Total raster:", len(quality_df))
print("Total missing pixels:", quality_df["missing_pixels"].sum())
print("Maximum missing percentage:",
      quality_df["missing_percentage"].max())

,date,valid_pixels,missing_pixels,missing_percentage
0,01,17373,0,0.0
1,02,17373,0,0.0
2,03,17373,0,0.0
3,04,17373,0,0.0
4,05,17373,0,0.0
5,06,17373,0,0.0
6,07,17373,0,0.0
7,08,17373,0,0.0
8,09,17373,0,0.0
9,10,17373,0,0.0


===== OVERALL QUALITY =====
Total raster: 30
Total missing pixels: 0
Maximum missing percentage: 0.0


## 7. Validation Summary

Seluruh 30 raster CHIRPS v3.0 SAT periode 1–30 September 2026 berhasil melewati pemeriksaan kelengkapan dan kualitas data.

Hasil validasi menunjukkan:
- Seluruh tanggal tersedia tanpa file yang hilang.
- Seluruh raster memiliki CRS EPSG:4326 dan resolusi sekitar 0.05°.
- Raster berhasil dipotong berdasarkan batas administratif lima provinsi Kalimantan.
- Setiap raster memiliki 17.373 piksel valid di dalam wilayah penelitian.
- Tidak ditemukan missing pixels di dalam wilayah penelitian pada seluruh periode pengamatan.

Dataset hasil cropping disimpan dalam format GeoTIFF pada `data/processed/chirps/daily/`.